# Modelling panel — descriptive EDA

Panel-level profiling on the supervised **pathogen-long** units from notebook `10` (B2/A3 contract).

| Item | Detail |
|------|--------|
| Input | `modelling_panel.parquet` + `modelling_panel_species.parquet` + `modelling_panel_manifest.json` |
| Grain | Observed VectorNet × pathogen: Tier 1 `(h3_cell, year, month, vector_group, pathogen_name)`; Tier 2 adds `country_iso2` |
| Primary target | `pathogen_within_5y` (forward ≤5y, same key × **this** pathogen) |
| Secondary / QA | `pathogen_within_10y`, ever / delay / same-month; source & geo shares for this pathogen |
| Spec | [`specs/03`](../../specs/03-spatial-temporal-matching-strategy.md) + [`specs/04`](../../specs/04-reference-data-and-panel-pipeline.md) |

**Prerequisite:** run [`../01_sources/10_merge_modelling_panel.ipynb`](../01_sources/10_merge_modelling_panel.ipynb) so the panel **and** species side table exist.

This notebook covers load → profile → missingness / outliers on the **raw** merge panel. Locked late priority species pivot, filters, and the modelling feature handoff live under [`../03_features/`](../03_features/). Stratify by `pathogen_name`.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from vector_guard.data import default_processed_dir, find_repo_root
from vector_guard.panel_features import SPECIES_LONG_COLS, UNIT_KEY_COLS, validate_species_long
from vector_guard.resolve import H3_RESOLUTION

REPO_ROOT = find_repo_root(Path.cwd())
PROCESSED_DIR = default_processed_dir(REPO_ROOT)
PANEL_PATH = PROCESSED_DIR / "modelling_panel.parquet"
SPECIES_PATH = PROCESSED_DIR / "modelling_panel_species.parquet"
MANIFEST_PATH = PROCESSED_DIR / "modelling_panel_manifest.json"

# Winsorize reference quantile for effort-proxy tail summaries (prep applies this in 03_features).
COUNT_WINSOR_Q = 0.99
COUNT_COLS = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
]

IDENTITY_COLS = [
    "spatial_tier",
    "h3_cell",
    "country_iso2",
    "year",
    "month",
    "vector_group",
    "pathogen_name",
]
TIER1_KEY = ["h3_cell", "year", "month", "vector_group", "pathogen_name"]
TIER2_KEY = ["country_iso2", "year", "month", "vector_group", "pathogen_name"]

# Primary supervised target (spec 04).
TARGET_COL = "pathogen_within_5y"
LABEL_COLS = [
    "pathogen_within_5y",
    "pathogen_within_10y",
    "pathogen_ever_before",
    "pathogen_ever_after",
    "pathogen_same_month",
    "pathogen_detection_count_within_5y",
    "pathogen_detection_count_within_10y",
    "years_to_first_pathogen_after",
    "years_since_last_pathogen_before",
]

# Forward-window QA columns (not modelling features — would leak label horizon).
SHARE_COLS = [
    "share_pathogen_exact_coords",
    "share_pathogen_subnational_centroid",
]

{
    "REPO_ROOT": str(REPO_ROOT),
    "PANEL_PATH": str(PANEL_PATH.relative_to(REPO_ROOT)),
    "SPECIES_PATH": str(SPECIES_PATH.relative_to(REPO_ROOT)),
    "H3_RESOLUTION": H3_RESOLUTION,
    "winsorize_cols": COUNT_COLS,
}


## Load panel, species side table, and merge manifest

Load the raw merge outputs for descriptive profiling. Do **not** require `vector_count_*` on the raw panel — priority species features are attached in [`../03_features/`](../03_features/) after filters via unit-grain pivot.


In [ ]:
if not PANEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PANEL_PATH}. Run notebooks/01_sources/10_merge_modelling_panel.ipynb first."
    )
if not SPECIES_PATH.exists():
    raise FileNotFoundError(
        f"Missing {SPECIES_PATH}. Run notebooks/01_sources/10_merge_modelling_panel.ipynb first."
    )

panel = pd.read_parquet(PANEL_PATH)
species_long = validate_species_long(pd.read_parquet(SPECIES_PATH))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}

assert "pathogen_name" in panel.columns
assert TARGET_COL in panel.columns
assert "virus_within_5y" not in panel.columns
assert "pathogen_name" not in species_long.columns
assert list(species_long.columns) == SPECIES_LONG_COLS
{
    "rows": len(panel),
    "cols": len(panel.columns),
    "species_rows": len(species_long),
    "species_units": int(species_long.loc[:, UNIT_KEY_COLS].drop_duplicates().shape[0]),
    "manifest_panel_rows": manifest.get("panel_rows"),
    "manifest_species_table_rows": manifest.get("species_table_rows"),
    "manifest_date_range": manifest.get("date_range"),
    "primary_label": manifest.get("primary_label"),
    "secondary_label": manifest.get("secondary_label"),
    "eligibility": manifest.get("eligibility"),
    "n_pathogens": manifest.get("n_pathogens"),
    "pathogen_positive_units": manifest.get("pathogen_positive_units"),
    "pathogen_positive_rate": manifest.get("pathogen_positive_rate"),
    "pathogen_within_10y_units": manifest.get("pathogen_within_10y_units"),
    "pathogen_ever_after_units": manifest.get("pathogen_ever_after_units"),
    "pathogen_ever_before_units": manifest.get("pathogen_ever_before_units"),
    "pathogen_same_month_units": manifest.get("pathogen_same_month_units"),
    "rows_by_spatial_tier": manifest.get("rows_by_spatial_tier"),
    "training_filter": manifest.get("training_filter"),
}


## Panel profile

Shape, dtypes, grain uniqueness, and basic value ranges.


In [ ]:
display(panel.dtypes.to_frame("dtype"))
display(panel.head(5))

tier1 = panel.loc[panel["spatial_tier"] == "h3"]
tier2 = panel.loc[panel["spatial_tier"] == "country"]

profile = {
    "n_rows": len(panel),
    "n_tier1_h3": len(tier1),
    "n_tier2_country": len(tier2),
    "dup_tier1_keys": int(tier1.duplicated(TIER1_KEY).sum()) if len(tier1) else 0,
    "dup_tier2_keys": int(tier2.duplicated(TIER2_KEY).sum()) if len(tier2) else 0,
    "year_min": int(panel["year"].min()),
    "year_max": int(panel["year"].max()),
    "n_h3_cells": int(panel["h3_cell"].nunique(dropna=True)),
    "n_countries": int(panel["country_iso2"].nunique(dropna=True)),
    "n_pathogens": int(panel["pathogen_name"].nunique()),
    "vector_groups": sorted(panel["vector_group"].dropna().unique().tolist()),
    "vector_present_all_one": bool((panel["vector_present"] == 1).all()),
    "pathogen_positive_5y": int(panel[TARGET_COL].sum()),
    "pathogen_positive_rate_5y": float(panel[TARGET_COL].mean()),
    "pathogen_within_10y": int(panel["pathogen_within_10y"].sum()),
    "pathogen_ever_after": int(panel["pathogen_ever_after"].sum()),
    "pathogen_ever_before": int(panel["pathogen_ever_before"].sum()),
    "pathogen_same_month": int(panel["pathogen_same_month"].sum()),
    "species_richness_mean": float(panel["species_richness"].mean()),
}
profile


In [ ]:
display(
    panel.groupby("pathogen_name", dropna=False)
    .agg(
        n_rows=("vector_present", "size"),
        within_5y=(TARGET_COL, "sum"),
        within_10y=("pathogen_within_10y", "sum"),
        ever_after=("pathogen_ever_after", "sum"),
        ever_before=("pathogen_ever_before", "sum"),
        same_month=("pathogen_same_month", "sum"),
        mean_vector_detections=("vector_detection_count", "mean"),
    )
    .assign(positive_rate_5y=lambda d: d["within_5y"] / d["n_rows"])
    .sort_values("positive_rate_5y", ascending=False)
)

display(
    panel.groupby(["vector_group", "pathogen_name"], dropna=False)
    .agg(
        n_rows=("vector_present", "size"),
        within_5y=(TARGET_COL, "sum"),
        within_10y=("pathogen_within_10y", "sum"),
    )
    .assign(positive_rate_5y=lambda d: d["within_5y"] / d["n_rows"])
)

display(
    panel.groupby("year", dropna=False)
    .agg(
        n_rows=("vector_present", "size"),
        within_5y=(TARGET_COL, "sum"),
        within_10y=("pathogen_within_10y", "sum"),
        ever_after=("pathogen_ever_after", "sum"),
    )
    .tail(20)
)


## Missing values

Identity, vector features, and binary labels should be complete on the supervised panel.

Expected nulls:
- `share_pathogen_*` — only defined when the 5y forward window has events of **this** pathogen
- `years_to_first_pathogen_after` / `years_since_last_pathogen_before` — null when that side has no event (`pathogen_ever_after` / `pathogen_ever_before` == 0)


In [ ]:
missing = (
    panel.isna()
    .sum()
    .rename("n_missing")
    .to_frame()
    .assign(pct=lambda d: 100.0 * d["n_missing"] / len(panel))
    .sort_values("n_missing", ascending=False)
)
display(missing.head(20))

share_qa = {
    "share_null_when_pathogen_0": float(
        panel.loc[panel[TARGET_COL] == 0, SHARE_COLS[0]].isna().mean()
    ),
    "share_null_when_pathogen_1": float(
        panel.loc[panel[TARGET_COL] == 1, SHARE_COLS[0]].isna().mean()
    ),
    "delay_after_null_when_ever_after_0": bool(
        panel.loc[panel["pathogen_ever_after"] == 0, "years_to_first_pathogen_after"].isna().all()
    ),
    "delay_before_null_when_ever_before_0": bool(
        panel.loc[panel["pathogen_ever_before"] == 0, "years_since_last_pathogen_before"].isna().all()
    ),
}
share_qa


## Outliers (vector effort proxies)

Detection and individual counts are right-skewed. Summarize tails and prepare winsor caps for **`COUNT_COLS` only** (group effort proxies). Do **not** winsorize priority `vector_count_*` species features. Do not drop units — rare pathogen positives must stay.


In [ ]:
outlier_summary = []
winsor_caps: dict[str, float] = {}
for col in COUNT_COLS:
    s = panel[col]
    cap = float(s.quantile(COUNT_WINSOR_Q))
    winsor_caps[col] = cap
    outlier_summary.append(
        {
            "column": col,
            "min": float(s.min()),
            "p50": float(s.median()),
            "p95": float(s.quantile(0.95)),
            f"p{int(COUNT_WINSOR_Q * 100)}": cap,
            "max": float(s.max()),
            "n_above_cap": int((s > cap).sum()),
            "n_pos_above_cap": int(
                ((panel[TARGET_COL] == 1) & (s > cap)).sum()
            ),
        }
    )

display(pd.DataFrame(outlier_summary))
winsor_caps
